# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Traffic fields are heavily right-skewed, checked before testing anything, not assumed.** Mean sitting several times above median is the tell: a handful of giant pages carry a huge share of total traffic, and plain Pearson correlation on raw values would be dominated by those few rows. Every test below uses Spearman (rank) correlation or grouped medians/weighted rates instead, per the skill.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

for col in ["impressions_90d", "sessions_90d", "clicks_90d", "word_count", "search_volume"]:
    d = df[col].dropna()
    heavy_tail = d.mean() > 2 * d.median()
    print(f"{col:16s} mean={d.mean():9.1f}  median={d.median():8.1f}  p95={d.quantile(0.95):9.1f}  max={d.max():9.1f}  heavy-tailed: {heavy_tail}")

impressions_90d  mean=   5200.4  median=   731.0  p95=  22996.5  max= 517715.0  heavy-tailed: True
sessions_90d     mean=     37.1  median=     7.0  p95=    166.0  max=   4345.0  heavy-tailed: True
clicks_90d       mean=     16.1  median=     1.0  p95=     69.0  max=   4178.0  heavy-tailed: True
word_count       mean=   3107.8  median=  2877.0  p95=   6173.0  max=   9546.0  heavy-tailed: False
search_volume    mean=    158.9  median=    10.0  p95=    390.0  max=  74000.0  heavy-tailed: True


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Test 1: "Longer content gets more traffic."** Claim in one sentence: pages with more words get more sessions. Test: Spearman correlation (traffic is heavy-tailed, see Section 1) plus grouped medians by `word_count_tier`.

In [2]:
t1 = df.dropna(subset=["word_count", "sessions_90d"])
rho1, p1 = spearmanr(t1["word_count"], t1["sessions_90d"])
medians1 = t1.groupby("word_count_tier", observed=True)["sessions_90d"].median()
counts1 = t1.groupby("word_count_tier", observed=True).size()
print(f"n={len(t1)}, spearman rho={rho1:.3f}, p={p1:.4g}")
print(pd.DataFrame({"median_sessions_90d": medians1, "n": counts1}))
print("\nVERDICT: CONFIRMED. Medians rise monotonically with word_count_tier "
      "(<1000: 1.0 -> 3500+: 22.0), every bucket well above the 50-row floor, "
      "and the direction matches the claim, not just statistically significant, practically visible.")

n=22301, spearman rho=0.393, p=0
                 median_sessions_90d      n
word_count_tier                            
1000-2000                        4.0   3780
2000-3500                        6.0  11263
3500+                           22.0   6285
<1000                            1.0    973

VERDICT: CONFIRMED. Medians rise monotonically with word_count_tier (<1000: 1.0 -> 3500+: 22.0), every bucket well above the 50-row floor, and the direction matches the claim, not just statistically significant, practically visible.


**Test 2: "Higher search-volume keywords get more clicks."** Claim: pages targeting higher-search-volume keywords should see more clicks. Test: Spearman correlation between `search_volume` and `clicks_90d`.

In [3]:
t2 = df.dropna(subset=["search_volume", "clicks_90d"])
rho2, p2 = spearmanr(t2["search_volume"], t2["clicks_90d"])
print(f"n={len(t2)}, spearman rho={rho2:.3f}, p={p2:.4g}")
print("\nVERDICT: MIXED. Statistically significant only because n is huge (27,532), "
      "but rho is -0.068, essentially flat and weakly the WRONG sign. Search volume alone "
      "is not a practically useful predictor of actual clicks in this data, a content team "
      "chasing high-search-volume keywords on this basis alone would be chasing noise.")

n=27532, spearman rho=-0.068, p=2.779e-29

VERDICT: MIXED. Statistically significant only because n is huge (27,532), but rho is -0.068, essentially flat and weakly the WRONG sign. Search volume alone is not a practically useful predictor of actual clicks in this data, a content team chasing high-search-volume keywords on this basis alone would be chasing noise.


**Test 3: "Transactional/commercial intent pages get higher CTR than informational."** Claim: pages built for someone ready to act should convert search interest into clicks better than purely informational pages. Test: median CTR by `main_intent`, plus the weighted rate (total clicks / total impressions, not the mean of per-page CTRs, per the skill's averaging trap).

In [4]:
t3 = df.dropna(subset=["main_intent", "ctr"])
t3 = t3[t3["ctr"] > 0]
grp3 = t3.groupby("main_intent", observed=True).agg(
    median_ctr=("ctr", "median"), n=("ctr", "size"),
    total_clicks=("clicks_90d", "sum"), total_impressions=("impressions_90d", "sum"))
grp3["weighted_ctr"] = grp3["total_clicks"] / grp3["total_impressions"]
print(grp3)
print("\nVERDICT: MIXED, with one bucket excluded on sample size. navigational (n=22) is below "
      "the ~30-row cross-cut floor, no verdict from it, whatever its rate says. Among the rest, "
      "the weighted rate does rank transactional above commercial above informational as the claim "
      "predicts, but median CTR shows commercial and informational tied (0.230 each), so the pattern "
      "is real but weak, not the clean story the claim implies.")

               median_ctr     n  total_clicks  total_impressions  weighted_ctr
main_intent                                                                   
commercial          0.230  2707         86947           26887474      0.003234
informational       0.230  9857        273106           90838630      0.003006
navigational        0.385    22           958             226457      0.004230
transactional       0.260  3550        118350           32214549      0.003674

VERDICT: MIXED, with one bucket excluded on sample size. navigational (n=22) is below the ~30-row cross-cut floor, no verdict from it, whatever its rate says. Among the rest, the weighted rate does rank transactional above commercial above informational as the claim predicts, but median CTR shows commercial and informational tied (0.230 each), so the pattern is real but weak, not the clean story the claim implies.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-linked test: does staleness actually predict decline?** This is the exact assumption FlyRank's own refresh-priority framing rests on (and the one my own ML-07 baseline rule leaned on): the longer a page sits untouched, the more likely it's declining. Test: decline rate by `days_since_last_update` bucket, with n shown next to every rate, per the skill.

In [5]:
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=[-1, 30, 90, 180, 100000],
                             labels=["0-30", "31-90", "91-180", "181+"])
flag_test = df.groupby("stale_bucket", observed=True)["is_declining_label"].agg(["mean", "count"])
flag_test.columns = ["decline_rate", "n"]
print(flag_test)
print("\nVERDICT: MIXED. The assumption holds in the middle: decline rate rises from 51.1% (0-30 days) "
      "to 61.1% (91-180 days), consistent with 'staler pages decline more.' But it reverses at the extreme: "
      "181+ day pages decline LEAST of all (47.1%), the opposite of what the flag assumes, on a bucket "
      "large enough to trust (n=174, above the ~50-row floor). Staleness is not a safe universal proxy for "
      "decline, it only points the right direction over part of its range, matching what ML-07's baseline "
      "rule found independently: a stale-based rule scored below the base rate on this same data.")

              decline_rate      n
stale_bucket                     
0-30              0.511377  20480
31-90             0.588571    175
91-180            0.611057   9171
181+              0.471264    174

VERDICT: MIXED. The assumption holds in the middle: decline rate rises from 51.1% (0-30 days) to 61.1% (91-180 days), consistent with 'staler pages decline more.' But it reverses at the extreme: 181+ day pages decline LEAST of all (47.1%), the opposite of what the flag assumes, on a bucket large enough to trust (n=174, above the ~50-row floor). Staleness is not a safe universal proxy for decline, it only points the right direction over part of its range, matching what ML-07's baseline rule found independently: a stale-based rule scored below the base rate on this same data.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**For a content team:** content length genuinely predicts traffic and is safe to act on (Test 1, CONFIRMED). Search volume alone is not a reliable prioritization signal, don't chase high-volume keywords expecting clicks to follow automatically (Test 2, MIXED and practically flat). Intent type matters some for CTR but only weakly and unevenly, exclude navigational entirely (too few rows) and treat the commercial-vs-informational gap as noise, not signal (Test 3, MIXED). Most importantly for how refresh priority actually gets decided: staleness alone is not a safe proxy for decline, it works in the middle of its range and inverts at the extreme, so any rule or model that leans on "hasn't been updated in a long time" as its main signal needs a second, independent signal alongside it, not staleness by itself.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.